<a href="https://colab.research.google.com/github/DhikshaSubash/market-sentiment-analysis/blob/main/04_sentiment_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Install dependencies
!pip install transformers torch pyspark -q

print("✅ Libraries installed!")

✅ Libraries installed!


In [2]:
# Cell 2 - Load FinBERT from HuggingFace
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("⏳ Loading FinBERT model... (first time takes ~1 min)")

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# Labels FinBERT uses
labels = ["positive", "negative", "neutral"]

print("✅ FinBERT loaded successfully!")
print(f"Labels: {labels}")

⏳ Loading FinBERT model... (first time takes ~1 min)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FinBERT loaded successfully!
Labels: ['positive', 'negative', 'neutral']


In [3]:
# Cell 3 - Test FinBERT on sample headlines
import torch.nn.functional as F

def get_sentiment(headline):
    inputs = tokenizer(
        headline,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)
    scores = probs[0].tolist()

    sentiment_scores = {
        "positive": round(scores[0], 4),
        "negative": round(scores[1], 4),
        "neutral":  round(scores[2], 4)
    }

    predicted_label = labels[scores.index(max(scores))]

    return predicted_label, sentiment_scores

# Test on sample headlines
test_headlines = [
    "Apple reports record breaking profits this quarter",
    "Tesla stock crashes after massive recall announcement",
    "Microsoft announces new product launch next month"
]

print("🧪 Testing FinBERT on sample headlines:\n")
for headline in test_headlines:
    label, scores = get_sentiment(headline)
    print(f"Headline : {headline}")
    print(f"Sentiment: {label.upper()}")
    print(f"Scores   : {scores}")
    print("-" * 60)

🧪 Testing FinBERT on sample headlines:

Headline : Apple reports record breaking profits this quarter
Sentiment: POSITIVE
Scores   : {'positive': 0.9257, 'negative': 0.0269, 'neutral': 0.0474}
------------------------------------------------------------
Headline : Tesla stock crashes after massive recall announcement
Sentiment: NEGATIVE
Scores   : {'positive': 0.0132, 'negative': 0.9407, 'neutral': 0.0461}
------------------------------------------------------------
Headline : Microsoft announces new product launch next month
Sentiment: NEUTRAL
Scores   : {'positive': 0.0787, 'negative': 0.0137, 'neutral': 0.9076}
------------------------------------------------------------


In [4]:
# Cell 4 - Load processed news from Drive
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("SentimentScoring") \
    .getOrCreate()

news_df = spark.read.parquet(
    "/content/drive/MyDrive/market_sentiment/processed/news"
)

print(f"✅ Loaded {news_df.count()} news records")
news_df.select("symbol", "headline").show(5, truncate=60)

Mounted at /content/drive
✅ Loaded 50 news records
+------+------------------------------------------------------------+
|symbol|                                                    headline|
+------+------------------------------------------------------------+
|  AAPL|                    openclaw-trading-assistant added to PyPI|
|  AAPL|Apple’s services flywheel spins at full speed, says Everc...|
|  AAPL|Morgan Stanley Is Doubling Down on Apple Stock Thanks to ...|
|  AAPL|   Apple (AAPL) Rated Sector Weight on Mixed Spending Trends|
|  AAPL|                                       finasys added to PyPI|
+------+------------------------------------------------------------+
only showing top 5 rows


In [5]:
# Cell 5 - Score all headlines with FinBERT
import pandas as pd

# Convert to Pandas for FinBERT processing
news_pd = news_df.select("symbol", "headline", "published_at").toPandas()

print(f"⏳ Scoring {len(news_pd)} headlines with FinBERT...\n")

results = []
for idx, row in news_pd.iterrows():
    label, scores = get_sentiment(row['headline'])
    results.append({
        "symbol":         row['symbol'],
        "headline":       row['headline'],
        "published_at":   row['published_at'],
        "sentiment":      label,
        "positive_score": scores['positive'],
        "negative_score": scores['negative'],
        "neutral_score":  scores['neutral']
    })

    # Progress update every 10 headlines
    if (idx + 1) % 10 == 0:
        print(f"  Scored {idx + 1}/{len(news_pd)} headlines...")

print(f"\n✅ All {len(results)} headlines scored!")

⏳ Scoring 50 headlines with FinBERT...

  Scored 10/50 headlines...
  Scored 20/50 headlines...
  Scored 30/50 headlines...
  Scored 40/50 headlines...
  Scored 50/50 headlines...

✅ All 50 headlines scored!


In [6]:
# Cell 6 - Convert results to Spark DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType
)

sentiment_schema = StructType([
    StructField("symbol",         StringType(), True),
    StructField("headline",       StringType(), True),
    StructField("published_at",   StringType(), True),
    StructField("sentiment",      StringType(), True),
    StructField("positive_score", FloatType(),  True),
    StructField("negative_score", FloatType(),  True),
    StructField("neutral_score",  FloatType(),  True),
])

sentiment_df = spark.createDataFrame(results, schema=sentiment_schema)

print("✅ Sentiment Spark DataFrame created!")
print(f"\nSentiment distribution:")
sentiment_df.groupBy("sentiment").count().orderBy("count", ascending=False).show()

print("\nSample scored headlines:")
sentiment_df.select(
    "symbol", "headline", "sentiment", "positive_score", "negative_score"
).show(5, truncate=50)

✅ Sentiment Spark DataFrame created!

Sentiment distribution:
+---------+-----+
|sentiment|count|
+---------+-----+
|  neutral|   30|
| negative|   11|
| positive|    9|
+---------+-----+


Sample scored headlines:
+------+--------------------------------------------------+---------+--------------+--------------+
|symbol|                                          headline|sentiment|positive_score|negative_score|
+------+--------------------------------------------------+---------+--------------+--------------+
|  AAPL|          openclaw-trading-assistant added to PyPI|  neutral|        0.0827|        0.0121|
|  AAPL|Apple’s services flywheel spins at full speed, ...|  neutral|         0.042|        0.0202|
|  AAPL|Morgan Stanley Is Doubling Down on Apple Stock ...| negative|        0.0137|         0.886|
|  AAPL|Apple (AAPL) Rated Sector Weight on Mixed Spend...| negative|        0.1315|        0.8316|
|  AAPL|                             finasys added to PyPI|  neutral|        0.0308| 

In [7]:
# Cell 7 - Sentiment summary per stock
from pyspark.sql.functions import round as spark_round, avg, count

summary_df = sentiment_df.groupBy("symbol").agg(
    count("headline").alias("total_articles"),
    spark_round(avg("positive_score"), 4).alias("avg_positive"),
    spark_round(avg("negative_score"), 4).alias("avg_negative"),
    spark_round(avg("neutral_score"),  4).alias("avg_neutral")
)

print("✅ Sentiment Summary Per Stock:\n")
summary_df.orderBy("avg_positive", ascending=False).show()

✅ Sentiment Summary Per Stock:

+------+--------------+------------+------------+-----------+
|symbol|total_articles|avg_positive|avg_negative|avg_neutral|
+------+--------------+------------+------------+-----------+
| GOOGL|            10|      0.3504|      0.1559|     0.4936|
|  MSFT|            10|      0.2768|      0.1406|     0.5826|
|  TSLA|            10|      0.2176|        0.33|     0.4523|
|  AMZN|            10|       0.204|      0.0151|     0.7809|
|  AAPL|            10|      0.1694|      0.2308|     0.5999|
+------+--------------+------------+------------+-----------+



In [8]:
# Cell 8 - Save sentiment scored data
sentiment_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/sentiment/scored_news"
)

summary_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/sentiment/summary"
)

print("✅ Sentiment data saved to Drive!")
print("\nFolder structure so far:")
print("""
market_sentiment/
├── raw/
│   ├── news/        ✅
│   └── prices/      ✅
├── processed/
│   ├── news/        ✅
│   └── prices/      ✅
└── sentiment/
    ├── scored_news/ ✅
    └── summary/     ✅
""")

✅ Sentiment data saved to Drive!

Folder structure so far:

market_sentiment/
├── raw/
│   ├── news/        ✅
│   └── prices/      ✅
├── processed/
│   ├── news/        ✅
│   └── prices/      ✅
└── sentiment/
    ├── scored_news/ ✅
    └── summary/     ✅

